In [2]:
!pip install torch torchvision jupyter
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torch
from torchvision import datasets, transforms

print("torch vision is imported")
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print(len(train_dataset), len(test_dataset))

torch vision is imported
60000 10000


## Step 2 — Build the model

In [4]:
import torch.nn as nn

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=64, bias=True)
  (2): ReLU()
  (3): Linear(in_features=64, out_features=10, bias=True)
)


## Step 3 - Training


In [5]:
import torch
from torch.utils.data import DataLoader

# dataloader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 2

for epoch in range(epochs):
    for images, labels in train_loader:
        
        # forward
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.24197934567928314
Epoch 2, Loss: 0.358875572681427


## Step 4 — Test the model

In [6]:
test_loader = DataLoader(test_dataset, batch_size=64)

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Accuracy:", accuracy)

Accuracy: 95.17


## Step 5 — Extract weights

In [7]:
import numpy as np

W1 = model[1].weight.detach().numpy()
b1 = model[1].bias.detach().numpy()

W2 = model[3].weight.detach().numpy()
b2 = model[3].bias.detach().numpy()

print(W1.shape, b1.shape)
print(W2.shape, b2.shape)

np.save("W1.npy", W1)
np.save("b1.npy", b1)
np.save("W2.npy", W2)
np.save("b2.npy", b2)

(64, 784) (64,)
(10, 64) (10,)


## Step 6 — Manual forward pass (NumPy)

In [18]:
def forward_numpy(x, W1, b1, W2, b2):
    h = np.dot(W1, x) + b1
    h = np.maximum(h, 0)   # ReLU
    y = np.dot(W2, h) + b2
    return y

# get one sample
image, label = test_dataset[0]

# flatten image (28x28 → 784)
x = image.view(-1).numpy()

# PyTorch output
with torch.no_grad():
    torch_output = model(image.unsqueeze(0))
    torch_pred = torch.argmax(torch_output).item()

# NumPy output
numpy_output = forward_numpy(x, W1, b1, W2, b2)
numpy_pred = np.argmax(numpy_output)

print("Label:", label)
print("PyTorch prediction:", torch_pred)
print("NumPy prediction:", numpy_pred)

Label: 7
PyTorch prediction: 7
NumPy prediction: 7


## Step 8 — Measure CPU inference time

In [19]:
import time

# prepare multiple samples
inputs = []
for i in range(1000):
    image, _ = test_dataset[i]
    x = image.view(-1).numpy()
    inputs.append(x)

# measure time
start = time.time()

for x in inputs:
    forward_numpy(x, W1, b1, W2, b2)

end = time.time()

print("CPU time for 1000 samples:", end - start)


CPU time for 1000 samples: 0.012011528015136719


In [20]:
print(W1.dtype, W2.dtype)

float32 float32


## Step 10 — Prepare data for FPGA

In [21]:
# pick one sample
image, label = test_dataset[0]

# flatten
x = image.view(-1).numpy().astype(np.float32)

print("x shape:", x.shape)
print("W1 shape:", W1.shape)

x shape: (784,)
W1 shape: (64, 784)


In [22]:
## Step 11 — Export binary files for Vitis HLS testbench
image, label = test_dataset[0]
x_test = image.view(-1).numpy().astype(np.float32)
y_expected = forward_numpy(x_test, W1, b1, W2, b2).astype(np.float32)

x_test.tofile("x.bin")
W1.astype(np.float32).tofile("W1.bin")
b1.astype(np.float32).tofile("b1.bin")
W2.astype(np.float32).tofile("W2.bin")
b2.astype(np.float32).tofile("b2.bin")
y_expected.tofile("y_expected.bin")

print("Exported binary files for HLS testbench")
print("Label:", label, "| Expected class:", np.argmax(y_expected))

Exported binary files for HLS testbench
Label: 7 | Expected class: 7


## Step 12 — Install hls4ml

In [24]:
try:
    import hls4ml
except ModuleNotFoundError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "hls4ml[profiling]", "--quiet"])
    import hls4ml

print("hls4ml version:", hls4ml.__version__)

hls4ml version: 1.3.0


## Step 13 — Prepare model for hls4ml

hls4ml expects a model whose first layer takes a flat 1-D input.  
We rebuild the same network **without** `nn.Flatten()` (hls4ml handles that internally), copy the trained weights, and verify it still predicts correctly.

In [8]:
import torch
import torch.nn as nn
import numpy as np

# Rebuild network without Flatten — input is already a flat 784-vector
model_hls = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

# Copy weights from the trained model (model[1] and model[3] because model had Flatten at [0])
model_hls[0].weight.data = model[1].weight.data.clone()
model_hls[0].bias.data   = model[1].bias.data.clone()
model_hls[2].weight.data = model[3].weight.data.clone()
model_hls[2].bias.data   = model[3].bias.data.clone()

model_hls.eval()

# Verify predictions still match
image, label = test_dataset[0]
x_flat = image.view(1, -1)           # shape (1, 784)

with torch.no_grad():
    pred_orig = torch.argmax(model(image.unsqueeze(0))).item()
    pred_hls  = torch.argmax(model_hls(x_flat)).item()

print(f"True label      : {label}")
print(f"Original model  : {pred_orig}")
print(f"model_hls       : {pred_hls}")
print("Weights match   :", pred_orig == pred_hls)

True label      : 7
Original model  : 7
model_hls       : 7
Weights match   : True


## Step 14 — Convert to HLS with hls4ml

`hls4ml` will generate fully pipelined HLS C++ with fixed-point arithmetic targeting the PYNQ-Z2 board.  
- `default_precision = 'fixed<16,6>'` → 16-bit fixed point, 6 integer bits (good for MNIST weights/activations).  
- `default_reuse_factor = 1` → maximum parallelism (each multiply gets its own DSP slice).  
- `backend = 'VivadoAccelerator'` → generates a standalone AXI-stream IP you can plug straight into Vivado.

In [ ]:
import hls4ml
import numpy as np
import os

# Build hls4ml config from the model
config = hls4ml.utils.config_from_pytorch_model(
    model_hls,
    input_shape=(784,),
    granularity='name',          # per-layer precision control
    default_precision='fixed<8,4>',
    default_reuse_factor=784       # serialize multiply over 784 inputs to reduce unrolling
)

# Save one sample/target pair for the generated testbench
image, _ = test_dataset[0]
x_sample = image.view(1, -1).numpy().astype(np.float32)   # shape (1, 784)
y_sample = model_hls(image.view(1, -1)).detach().numpy().astype(np.float32)
np.save('x_sample.npy', x_sample)
np.save('y_sample.npy', y_sample)

# Vitis HLS on Windows rejects project paths containing spaces
hls_output_dir = r"C:\Xilinx\Projects\hls4ml_mnist"
os.makedirs(hls_output_dir, exist_ok=True)

hls_model = hls4ml.converters.convert_from_pytorch_model(
    model_hls,
    hls_config=config,
    output_dir=hls_output_dir,
    part='xc7z020clg400-1',            # PYNQ-Z2 FPGA
    input_data_tb='x_sample.npy',
    output_data_tb='y_sample.npy'
)


Interpreting Model ...
Topology:
Layer name: _0, layer type: Dense, input shape: [[None, 784]]
Layer name: _1, layer type: Activation, input shape: [[None, 64]]
Layer name: _2, layer type: Dense, input shape: [[None, 64]]


In [17]:
import numpy as np
import os

# Provide one sample so hls4ml knows the input shape
image, _ = test_dataset[0]
x_sample = image.view(1, -1).numpy().astype(np.float32)   # shape (1, 784)
y_sample = model_hls(image.view(1, -1)).detach().numpy().astype(np.float32)
np.save('x_sample.npy', x_sample)
np.save('y_sample.npy', y_sample)

# Use a no-space output directory for Vitis HLS on Windows
hls_output_dir = r"C:\Xilinx\Projects\hls4ml_mnist"
os.makedirs(hls_output_dir, exist_ok=True)

hls_model = hls4ml.converters.convert_from_pytorch_model(
    model_hls,
    hls_config=config,
    output_dir=hls_output_dir,
    backend='VivadoAccelerator',
    board='pynq-z2',
    input_data_tb='x_sample.npy',
    output_data_tb='y_sample.npy'
)

print("HLS project written to:", hls_output_dir)
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_layer_names=True, to_file='hls_model.png')

Interpreting Model ...
Topology:
Layer name: _0, layer type: Dense, input shape: [[None, 784]]
Layer name: _1, layer type: Activation, input shape: [[None, 64]]
Layer name: _2, layer type: Dense, input shape: [[None, 64]]
HLS project written to: C:\Xilinx\Projects\hls4ml_mnist
Failed to import pydot. You must install pydot and graphviz for `pydotprint` to work.


## Step 15 — Compile and simulate in software (C-sim)

Before running Vitis HLS synthesis, verify that the fixed-point C++ model gives the same predictions as PyTorch.  
`hls_model.compile()` builds the generated C++ into a shared library and runs it locally — no FPGA needed.

In [19]:
import os
import platform
import shutil
import subprocess
from types import MethodType

msys2_bin = r"C:\msys64\ucrt64\bin"
if platform.system() == "Windows" and os.path.isdir(msys2_bin):
    if msys2_bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = msys2_bin + os.pathsep + os.environ.get("PATH", "")
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(msys2_bin)

can_compile = shutil.which("g++") is not None

if can_compile:
    if platform.system() == "Windows":
        # On Windows, os.rename() fails if dest exists. 
        # Monkey-patch it to use os.replace() which handles overwrites.
        _original_rename = os.rename
        def _rename_windows_safe(src, dst):
            if os.path.exists(dst):
                os.remove(dst)
            return _original_rename(src, dst)
        os.rename = _rename_windows_safe
    
        def _compile_with_bash(self, model):
            ret_val = subprocess.run(
                [r"C:\Program Files\Git\bin\bash.exe", "build_lib.sh"],
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                cwd=model.config.get_output_dir(),
            )
            if ret_val.returncode != 0:
                print(ret_val.stdout)
                raise Exception(f'Failed to compile project "{model.config.get_project_name()}"')
            return '{}/firmware/{}-{}.so'.format(
                model.config.get_output_dir(), model.config.get_project_name(), model.config.get_config_value('Stamp')
            )

        hls_model.config.backend.compile = MethodType(_compile_with_bash, hls_model.config.backend)

    hls_model.compile()   # builds C++ shared library locally (requires g++)

    # Run software prediction with fixed-point model
    y_hls = hls_model.predict(x_sample)   # returns numpy array shape (1, 10)

    print("PyTorch logits  :", model_hls(image.view(1,-1)).detach().numpy().round(3))
    print("hls4ml logits   :", y_hls.round(3))
    print("PyTorch class   :", int(model_hls(image.view(1,-1)).argmax()))
    print("hls4ml  class   :", int(y_hls.argmax()))
    print("True label      :", label)

    max_err = float(abs(y_hls - model_hls(image.view(1,-1)).detach().numpy()).max())
    print(f"Max fixed-pt err: {max_err:.4f}  ({'OK' if max_err < 0.5 else 'check precision'})")
else:
    print("Skipping hls_model.compile() because no g++ compiler is available on this machine.")
    print("The hls4ml project was generated successfully; run this step on a machine with g++ to perform C-sim.")

RecursionError: maximum recursion depth exceeded

In [ ]:
# Ensure MSYS2 tools (tee, g++) are available and install if missing
import os, shutil, subprocess, platform
msys2_usr = r"C:\msys64\usr\bin"
msys2_ucrt = r"C:\msys64\ucrt64\bin"
for p in (msys2_ucrt, msys2_usr):
    if os.path.isdir(p) and p not in os.environ.get('PATH',''):
        os.environ['PATH'] = p + os.pathsep + os.environ.get('PATH','')
print('PATH updated with MSYS2 bins')
print('tee available:', shutil.which('tee'))
print('g++ available:', shutil.which('g++'))
# If missing, try installing via MSYS2 pacman (requires MSYS2 installed)
if shutil.which('tee') is None or shutil.which('g++') is None:
    bash = os.path.join('C:', 'msys64', 'usr', 'bin', 'bash.exe')
    if os.path.isfile(bash):
        print('Installing coreutils and gcc via MSYS2 pacman (may take a few minutes)')
        subprocess.run([bash, '-lc', 'pacman -Sy --noconfirm coreutils mingw-w64-ucrt-x86_64-gcc'], check=False)
    else:
        print('MSYS2 bash not found; please install MSYS2 manually and run pacman commands.')
print('After install — tee:', shutil.which('tee'), 'g++:', shutil.which('g++'))

## Step 16 — Run Vitis HLS synthesis

This step calls Vivado HLS to synthesize the generated C++ into an RTL IP core.  
It takes **5–15 minutes**. When done it prints resource utilization (DSPs, LUTs, FFs) and the estimated clock period.

> **Note:** Vivado/Vitis HLS must be installed and on your PATH for this to work.  
> On Windows run `source /opt/Xilinx/Vitis_HLS/2022.2/settings64.sh` (or the equivalent `.bat`) before launching Jupyter.

In [15]:
import os
import shutil
from types import MethodType
from hls4ml.report import parse_vivado_report
import subprocess

# Ensure Xilinx tools are discoverable inside this Python process
xilinx_bins = [
    r"C:\Xilinx\Vitis_HLS\2024.2\bin",
    r"C:\Xilinx\Vivado\2024.2\bin",
    r"C:\Xilinx\Vitis\2024.2\bin",
    r"C:\msys64\usr\bin",            # tee.exe used by vitis_hls.bat
    r"C:\msys64\ucrt64\bin",         # GCC runtime libs
]
for p in xilinx_bins:
    if os.path.isdir(p) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = p + os.pathsep + os.environ.get("PATH", "")

print("vitis_hls found:", shutil.which("vitis_hls") is not None)
print("vivado found   :", shutil.which("vivado") is not None)
print("tee found      :", shutil.which("tee") is not None)

# hls4ml Vivado backend calls legacy `vivado_hls`; Xilinx 2024.2 provides `vitis_hls`.
vitis_hls_bat = r"C:\Xilinx\Vitis_HLS\2024.2\bin\vitis_hls.bat"

if os.path.isfile(vitis_hls_bat):
    def _build_with_vitis_hls_verbose(
        self,
        model,
        reset=False,
        csim=True,
        synth=True,
        cosim=False,
        validation=False,
        export=False,
        vsynth=False,
        fifo_opt=False,
    ):
        curr_dir = os.getcwd()
        output_dir = model.config.get_output_dir()
        os.chdir(output_dir)
        
        cmd = (
            f'"{vitis_hls_bat}" -f build_prj.tcl "reset={reset} '
            f'csim={csim} '
            f'synth={synth} '
            f'cosim={cosim} '
            f'validation={validation} '
            f'export={export} '
            f'vsynth={vsynth} '
            f'fifo_opt={fifo_opt}"'
        )
        
        print(f"Running: {cmd}")
        ret = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print("=== VITIS HLS STDOUT ===")
        print(ret.stdout)
        if ret.stderr:
            print("=== VITIS HLS STDERR ===")
            print(ret.stderr)
        print(f"vitis_hls return code: {ret.returncode}")
        
        os.chdir(curr_dir)
        
        # Only try to parse report if synthesis succeeded
        if ret.returncode == 0:
            return parse_vivado_report(output_dir)
        else:
            print("Synthesis failed - returning None")
            return None

    hls_model.config.backend.build = MethodType(_build_with_vitis_hls_verbose, hls_model.config.backend)

# If output_dir already exists, force-write clean project files so build_prj.tcl is present
output_dir = hls_model.config.get_output_dir()
build_tcl = os.path.join(output_dir, "build_prj.tcl")
if not os.path.isfile(build_tcl):
    if os.path.isdir(output_dir):
        shutil.rmtree(output_dir, ignore_errors=True)
    os.makedirs(output_dir, exist_ok=True)
    hls_model.write()

# Fix Vitis HLS 2024.2 compatibility: remove deprecated -maximum_size parameter
# which is not supported in 2024.2's config_array_partition command
if os.path.isfile(build_tcl):
    with open(build_tcl, 'r') as f:
        tcl_content = f.read()
    
    # Comment out or remove the problematic line
    tcl_content = tcl_content.replace(
        'catch {config_array_partition -maximum_size $maximum_size}',
        '# catch {config_array_partition -maximum_size $maximum_size}  # Removed for Vitis HLS 2024.2 compatibility'
    )
    
    with open(build_tcl, 'w') as f:
        f.write(tcl_content)
    print("Fixed build_prj.tcl for Vitis HLS 2024.2 compatibility")

report = hls_model.build(
    csim=False,       # skip C-sim (already done above)
    synth=True,       # run HLS synthesis -> RTL
    cosim=False,      # skip RTL co-sim (slow)
    export=True       # package as Vivado IP .zip
)

# Print synthesis summary only if the Vivado project was generated
project_name = hls_model.config.get_project_name()
project_dir = os.path.join(output_dir, f"{project_name}_prj")

if os.path.isdir(project_dir):
    print("\nSynthesis project directory found")
    ip_dir = os.path.join(output_dir, f"{project_name}_prj", "solution1", "impl", "ip")
    if os.path.isdir(ip_dir):
        print(f"IP files located at: {ip_dir}")
        ip_files = os.listdir(ip_dir)
        print(f"IP directory contents: {ip_files[:5]}...")
else:
    print(f"\nSynthesis project folder not found: {project_dir}")
    print("Check vitis_hls output above for synthesis errors.")

vitis_hls found: True
vivado found   : True
tee found      : True
Fixed build_prj.tcl for Vitis HLS 2024.2 compatibility
Running: "C:\Xilinx\Vitis_HLS\2024.2\bin\vitis_hls.bat" -f build_prj.tcl "reset=False csim=False synth=True cosim=False validation=False export=True vsynth=False fifo_opt=False"
=== VITIS HLS STDOUT ===

****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2024.2 (64-bit)
  **** SW Build 5238294 on Nov  8 2024
  **** IP Build 5239520 on Sun Nov 10 16:12:51 MST 2024
  **** SharedData Build 5239561 on Fri Nov 08 14:39:27 MST 2024
  **** Start of session at: Wed Apr 29 20:00:23 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

source C:/Xilinx/Vitis/2024.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] For user 'Mr. Moreira' on host 'laptop-spte7rgf' (Windows NT_amd64 version 10.0) on Wed Apr 29 20:00:24 +0200 2026
INFO: [HLS 200-10] In directory 'C:/Xilinx/Pr

## Step 17 — What to do next (Vivado + PYNQ)

After synthesis completes, `hls4ml_mnist/` contains a packaged Vivado IP.  
Follow these steps to deploy it on the PYNQ-Z2:

1. **Add IP to Vivado catalog** — open your existing block design, click *IP Settings → Add Repository*, point it at `hls4ml_mnist/myproject_prj/solution1/impl/ip`.  
2. **Replace the old `nn_forward` IP** with the new `myproject` IP (AXI-stream interface, not AXI-lite pointer interface).  
3. **Add DMA** — the hls4ml IP uses AXI-Stream. Add a `AXI DMA` block; connect S2MM/MM2S to the IP's input/output streams.  
4. **Regenerate bitstream** and copy the new `.bit` / `.hwh` pair to the PYNQ board.  
5. **Use the PYNQ DMA notebook template** (`pynq_hls4ml_inference.ipynb`, see next cell) — it sends data via DMA instead of manually poking AXI-Lite registers.